# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimahmahmood/flyrank_ml_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose a Decision Tree because my task is to classify content as declining or not declining. A Decision Tree can learn relationships between content performance signals and the declining label. I chose it because the model is relatively easy to interpret, so I can inspect which features are useful for the prediction rather than only looking at the final score.


In [18]:
!git clone https://github.com/fatimahmahmood/flyrank_ml_internship.git

fatal: destination path 'flyrank_ml_internship' already exists and is not an empty directory.


In [19]:
import pandas as pd
df = pd.read_csv(path)
print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [20]:
# Check the number of rows and unique clients
print("Rows:", len(df))
print("Unique clients:", df["client_id"].nunique())
# Check how many rows each client has
print("\nRows per client:")
print(df["client_id"].value_counts().head(10))

Rows: 30000
Unique clients: 32

Rows per client:
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031
Name: count, dtype: int64


## 2. Split design

I used a grouped train-test split based on client_id. This means that content from the same client does not appear in both the training and test sets. I chose this design because clients may have different content patterns, and allowing the same client in both sets could give the model an unfair advantage. The grouped split provides a more honest estimate of how well the model generalizes to unseen clients.

The target variable is whether content is declining trend_direction = "down". I convert this into a binary label:

* 1 = declining
* 0 = not declining


In [21]:
# Check the target we want to predict
print(df["trend_direction"].value_counts(dropna=False))
print("\nDeclining vs not declining:")
print(df["trend_direction"]
    .eq("down")
    .value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining vs not declining:
trend_direction
True     16262
False    13738
Name: count, dtype: int64


In [22]:
# Create binary target
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
# Check target distribution
print(df["is_declining"].value_counts())
features = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days"]
# Remove rows with missing values
model_df = df[features + ["is_declining", "client_id"]].dropna()
print("Rows after cleaning:", len(model_df))
print(model_df.head())

is_declining
1    16262
0    13738
Name: count, dtype: int64
Rows after cleaning: 22301
   days_since_last_update  impressions_90d  clicks_90d  pageviews_90d  \
0                      20             3803          29             22   
1                      25            15320           7             10   
2                      20            12581          11             14   
4                      14            19140          24            177   
5                      20             3970           1              4   

   sessions_90d  users_90d   ctr  avg_position  word_count  content_age_days  \
0            17         16  0.76          10.6      3221.0               187   
1             9          9  0.05          20.3      2481.0               445   
2            11         11  0.09          36.5      3515.0               141   
4           145        144  0.13          44.0      2803.0               263   
5             5          5  0.03           8.5      3080.0               

In [23]:
from sklearn.model_selection import GroupShuffleSplit

X = model_df[features]
y = model_df["is_declining"]
groups = model_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42)

train_idx, test_idx = next(
    gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

test_df = model_df.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTrain clients:", len(groups.iloc[train_idx].unique()))
print("Test clients:", len(groups.iloc[test_idx].unique()))

Train rows: 17223
Test rows: 5078

Train clients: 25
Test clients: 7


## 3. Train + compare vs my baseline

I trained a Decision Tree using the training clients and evaluated it on the unseen test clients. I used the same underlying content data for the comparison. The target is whether the content is declining. I use F1 as the evaluation metric because the task needs a balance between identifying declining content and avoiding incorrect declining predictions.


In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score)
# Create the Decision Tree
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42)

# Train the model
model.fit(X_train, y_train)

# Predict on unseen test clients
y_pred = model.predict(X_test)

# Calculate metrics
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print("Decision Tree results")
print("---------------------")
print("F1:", round(f1, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("Accuracy:", round(accuracy, 3))

Decision Tree results
---------------------
F1: 0.628
Precision: 0.571
Recall: 0.699
Accuracy: 0.569


In [25]:
# Recreate the Week 4 baseline on the test data.
# Volume thresholds are learned from the training data only.

q1, q2, q3 = X_train["impressions_90d"].quantile([0.25, 0.50, 0.75])

baseline_test = test_df.copy()

# Create volume buckets using training-data thresholds
baseline_test["volume_points"] = 0

baseline_test.loc[
    baseline_test["impressions_90d"] >= q1,
    "volume_points"] = 1

baseline_test.loc[
    baseline_test["impressions_90d"] >= q2,
    "volume_points"] = 2

baseline_test["stale_points"] = 0

baseline_test.loc[
    baseline_test["days_since_last_update"].between(91, 180),
    "stale_points"] = 2

baseline_test.loc[
    baseline_test["days_since_last_update"] >= 181,
    "stale_points"] = 1
baseline_test["baseline_score"] = (
    baseline_test["stale_points"]
    + baseline_test["volume_points"])
baseline_test["baseline_pred"] = (
    baseline_test["baseline_score"] >= 2
).astype(int)
baseline_f1 = f1_score(
    y_test,
    baseline_test["baseline_pred"])
baseline_precision = precision_score(
    y_test,
    baseline_test["baseline_pred"],
    zero_division=0)
baseline_recall = recall_score(
    y_test,
    baseline_test["baseline_pred"],
    zero_division=0)
print("Week 4 Baseline results")
print("-----------------------")
print("F1:", round(baseline_f1, 3))
print("Precision:", round(baseline_precision, 3))
print("Recall:", round(baseline_recall, 3))

Week 4 Baseline results
-----------------------
F1: 0.513
Precision: 0.495
Recall: 0.533


### Model vs baseline

The Decision Tree achieved an F1 score of 0.628 on the test clients, compared with 0.513 for the Week 4 baseline. Precision increased from 0.495 to 0.571, while recall increased from 0.533 to 0.699. On this test split, the Decision Tree therefore performed better than the baseline on all three measured metrics.

The comparison is directional and applies to this grouped-by-client test split. The higher recall means the Decision Tree identified more of the content labeled as declining, while the higher precision means a larger share of its declining predictions were correct.


In [26]:
comparison = pd.DataFrame({
    "Method": [
        "Week 4 Baseline",
        "Decision Tree" ],
    "F1": [
        baseline_f1,
        f1],
    "Precision": [
        baseline_precision,
        precision],
    "Recall": [
        baseline_recall,
        recall]
})
comparison.round(3)

,Method,F1,Precision,Recall
0,Week 4 Baseline,0.513,0.495,0.533
1,Decision Tree,0.628,0.571,0.699


## 4. Errors and interpretation

I examined the predictions made by the Decision Tree on the unseen test clients. I focused on false positives, where the model predicted declining content when the observed label was not declining, and false negatives, where the model missed content labeled as declining. I also examined feature importance to understand which signals the model relied on most.


In [27]:
# Create a table of test predictions and actual outcomes
error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["predicted"] = y_pred

# Identify prediction errors
error_df["error_type"] = "correct"

error_df.loc[
    (error_df["actual"] == 0) & (error_df["predicted"] == 1),
    "error_type"
] = "false_positive"

error_df.loc[
    (error_df["actual"] == 1) & (error_df["predicted"] == 0),
    "error_type"
] = "false_negative"

print(error_df["error_type"].value_counts())

error_df[error_df["error_type"] != "correct"].head(10)

error_type
correct           2889
false_positive    1391
false_negative     798
Name: count, dtype: int64


,days_since_last_update,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,ctr,avg_position,word_count,content_age_days,actual,predicted,error_type
1,25,15320,7,10,9,9,0.05,20.3,2481.0,445,1,0,false_negative
13,103,307,0,4,4,4,0.00,39.8,1342.0,238,0,1,false_positive
26,13,2426,3,9,9,9,0.12,30.0,2686.0,300,0,1,false_positive
36,20,371,5,6,5,5,1.35,5.4,2510.0,187,0,1,false_positive
39,104,4,0,1,1,1,0.00,36.3,3666.0,348,1,0,false_negative
51,8,2,0,1,3,3,0.00,7.5,2756.0,126,1,0,false_negative
56,20,16,0,2,2,2,0.00,4.6,3158.0,145,0,1,false_positive
64,8,2639,3,8,6,6,0.11,7.2,2808.0,106,0,1,false_positive
78,92,59,0,3,3,3,0.00,8.7,1589.0,174,0,1,false_positive
82,13,1810,8,12,8,7,0.44,8.3,2793.0,348,0,1,false_positive


In [28]:
# Check which features the Decision Tree relied on most
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_})
feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False)
feature_importance

,feature,importance
1,impressions_90d,0.703706
9,content_age_days,0.091490
6,ctr,0.091240
7,avg_position,0.044366
2,clicks_90d,0.029457
3,pageviews_90d,0.018426
0,days_since_last_update,0.010481
8,word_count,0.009970
4,sessions_90d,0.000865
5,users_90d,0.000000


### Error analysis and interpretation

The Decision Tree made 2,889 correct predictions, 1,391 false positives, and 798 false negatives. It made more false positives than false negatives, so it sometimes flagged content as declining when it was not.

The model relied most on `impressions_90d` (0.704). The next most important features were `content_age_days` (0.091) and `ctr` (0.091). `days_since_last_update`, which was used in my Week 4 baseline, had much lower importance (0.010).

This shows that the model used more information than the Week 4 baseline. These results describe what the model used for prediction and do not mean that these features cause content to decline. The results are based on this test split.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.